# Objective: trace storage state of charge

Demonstrate charging, discharging, bounds, and terminal state of charge with a four-hour fixture. Students submit capacities; the evaluator dispatches storage.

In [ ]:
from pathlib import Path
import json, os, sys

TUTORIAL_ROOT = Path(os.environ.get('TUTORIAL_ROOT', Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('TUTORIAL_OUTPUT_ROOT', TUTORIAL_ROOT / 'outputs')).resolve(); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
power_mw, energy_mwh = 5.0, 20.0
initial_soc = terminal_target = 0.5 * energy_mwh
charge_mw = [5.0, 0.0, 0.0, 0.0]
discharge_mw = [0.0, 0.0, 3.0, 2.0]


In [ ]:
print('Environment report')
print({'python': sys.version.split()[0], 'storage_model': 'ideal-efficiency educational fixture', 'fixture_first': True})


In [ ]:
soc = [initial_soc]
for charge, discharge in zip(charge_mw, discharge_mw):
    assert 0 <= charge <= power_mw and 0 <= discharge <= power_mw
    soc.append(soc[-1] + charge - discharge)
assert all(0 <= value <= energy_mwh for value in soc)
assert all(discharge <= prior_soc for discharge, prior_soc in zip(discharge_mw, soc[:-1]))
assert soc[-1] >= terminal_target
print('SOC trajectory (MWh):', soc); print('terminal SOC (MWh):', soc[-1])


In [ ]:
OUTPUT = OUTPUT_ROOT / 'storage_soc.json'
payload = {'power_capacity_mw': power_mw, 'energy_capacity_mwh': energy_mwh, 'charge_mw': charge_mw, 'discharge_mw': discharge_mw, 'soc_mwh': soc, 'terminal_soc_mwh': soc[-1], 'terminal_target_mwh': terminal_target}
OUTPUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('OUTPUT:', OUTPUT)
